In [ ]:
include("../Envs/Env.jl")
include("../Algorithms/PPO-RNN.jl")

## 1. Prepare Environment

In [ ]:
Threads.nthreads()

In [ ]:
pomdp = LightDark1D()
pomdp_name = "LightDark"
bool_full_observability = false
env = Env(pomdp, bool_full_observability)
action_space = GetActionSpace(env)
function create_env()
    return Env(pomdp, bool_full_observability)
end

# define convert_o function
# function POMDPs.convert_o(T::Type{<:AbstractArray}, o::Int64, m::LightDark1D)
#     vec = zeros(Float32, 3)
#     vec[o] = 1.0f0
#     return vec
# end

# define process action function
function process_action(action_index::Int, action_space::UnitRange{Int})
    len = length(action_space)
    # idx = action - first(action_space) + 1
    # (idx < 1 || idx > len) && error("Action $action not in action space")
    # Find the index of the action value in the action space\n",
    
    (action_index < 1 || action_index > len) && error("Action index $action_index not in range 1:$len")

    onehot = zeros(Float32, len)
    onehot[action_index] = 1.0f0
    return onehot
end

#     return Float32.(action_space[action_index])
# end

In [ ]:
process_action(3, action_space)

## 2. Prepare Parameters

In [ ]:
state_dim = GetObsDim(env)
action_dim = 1
layer_size = 64
rnn_hidden_size = 64
gamma = discount(pomdp)
training_episodes = 10000
batch_size = 1024

## 3. Prepare PPO-RNN agent

In [ ]:
# if want to use gpu, need to uncomment the below line, and use device=Flux.gpu
# using CUDA

agent = PPORNNAgent(action_space, state_dim;
    hidden_dim=layer_size, 
    rnn_hidden_size=rnn_hidden_size, 
    batch_size=batch_size, 
    device=Flux.cpu) 

## 4. Train

In [ ]:
# 训练
rewards, losses, evals = train!(create_env, agent, training_episodes)

## 5. Evaluation

In [ ]:
evaluate(env, agent; num_episodes=10000, max_steps=100) 

## (Todo) Save or plot the data from Train (rewards, losses, evals)